In [33]:
# ============================================================
# STUDENT READINESS AI
# PRODUCTION-READY KAGGLE TRAINING PIPELINE
# ============================================================

!pip install -q catboost

import os
import json
import glob
import warnings
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

print("Environment ready.")

Environment ready.


In [34]:
import os

print(os.listdir('/kaggle/input'))

['datasets']


In [35]:
import os

dataset_path = "/kaggle/input/datasets"

print(os.listdir(dataset_path))

['jannatulferdaues']


In [36]:
import os

dataset_path = "/kaggle/input/datasets/jannatulferdaues"

print(os.listdir(dataset_path))

['ghid-student-dataset']


In [37]:
import os

dataset_path = "/kaggle/input/datasets/jannatulferdaues/ghid-student-dataset"

print(os.listdir(dataset_path))

['student_dataset.xlsx']


In [38]:
DATA_PATH = "/kaggle/input/datasets/jannatulferdaues/ghid-student-dataset/student_dataset.xlsx"

df = pd.read_excel(DATA_PATH)

# Drop the stray "Unnamed: 0" index column that Excel exports sometimes carry
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Shape:", df.shape)
df.head()

Shape: (55000, 11)


,Country,University,Country_HDI,Country_Life_Expectancy,Education_Level,Field_of_Study,Language_Proficiency,Health_Quality_Score_1to5,Stress_Level_1to5,Monthly_Earnings_Savings_USD,Readiness_Score_0to100
0,Afghanistan,Afghan University,0.498,57.950001,Master's,Natural Sciences,Native/Fluent,2,2,1316.37,61
1,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Education,Native/Fluent,2,4,1892.79,70
2,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Social Sciences & Psychology,Native/Fluent,2,4,2030.31,69
3,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Engineering & Technology,Native/Fluent,3,2,1646.31,64
4,Afghanistan,Afghan University,0.498,57.950001,Master's,"Arts, Humanities & Design",Beginner (A1-A2),2,3,1299.43,52


In [39]:
# ============================================================
# DATA CLEANING
# ============================================================

# Remove Excel-generated empty index column
if "Unnamed: 0" in df.columns:
    df.drop(
        columns=["Unnamed: 0"],
        inplace=True
    )

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

# Remove duplicate records
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal :", after)
print("Duplicates removed            :", before - after)

print("\nMissing values:")
display(df.isnull().sum())

print("\nFinal shape:", df.shape)

Rows before duplicate removal: 55000
Rows after duplicate removal : 55000
Duplicates removed            : 0

Missing values:


Country                         0
University                      0
Country_HDI                     0
Country_Life_Expectancy         0
Education_Level                 0
Field_of_Study                  0
Language_Proficiency            0
Health_Quality_Score_1to5       0
Stress_Level_1to5               0
Monthly_Earnings_Savings_USD    0
Readiness_Score_0to100          0
dtype: int64


Final shape: (55000, 11)


In [40]:
# ============================================================
# TARGET AND FEATURES
# ============================================================

TARGET = "Readiness_Score_0to100"

FEATURES = [
    "Country",
    "University",
    "Country_HDI",
    "Country_Life_Expectancy",
    "Education_Level",
    "Field_of_Study",
    "Language_Proficiency",
    "Health_Quality_Score_1to5",
    "Stress_Level_1to5",
    "Monthly_Earnings_Savings_USD"
]

CATEGORICAL_FEATURES = [
    "Country",
    "University",
    "Education_Level",
    "Field_of_Study",
    "Language_Proficiency"
]

NUMERICAL_FEATURES = [
    "Country_HDI",
    "Country_Life_Expectancy",
    "Health_Quality_Score_1to5",
    "Stress_Level_1to5",
    "Monthly_Earnings_Savings_USD"
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Target:", TARGET)
print("Number of features:", len(FEATURES))

display(X.head())

Target: Readiness_Score_0to100
Number of features: 10


,Country,University,Country_HDI,Country_Life_Expectancy,Education_Level,Field_of_Study,Language_Proficiency,Health_Quality_Score_1to5,Stress_Level_1to5,Monthly_Earnings_Savings_USD
0,Afghanistan,Afghan University,0.498,57.950001,Master's,Natural Sciences,Native/Fluent,2,2,1316.37
1,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Education,Native/Fluent,2,4,1892.79
2,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Social Sciences & Psychology,Native/Fluent,2,4,2030.31
3,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Engineering & Technology,Native/Fluent,3,2,1646.31
4,Afghanistan,Afghan University,0.498,57.950001,Master's,"Arts, Humanities & Design",Beginner (A1-A2),2,3,1299.43


In [41]:
# ============================================================
# NUMERICAL VALIDATION
# ============================================================

# Keep readiness score inside expected range
df[TARGET] = df[TARGET].clip(0, 100)

# Prevent impossible numerical values
df["Country_HDI"] = df["Country_HDI"].clip(0, 1)

df["Health_Quality_Score_1to5"] = (
    df["Health_Quality_Score_1to5"].clip(1, 5)
)

df["Stress_Level_1to5"] = (
    df["Stress_Level_1to5"].clip(1, 5)
)

df["Monthly_Earnings_Savings_USD"] = (
    df["Monthly_Earnings_Savings_USD"].clip(lower=0)
)

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Validation complete.")

Validation complete.


In [42]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

def preprocess_data(data):

    data = data[FEATURES].copy()

    # Log transformation for highly-skewed earnings
    data["Earnings_Log"] = np.log1p(
        data["Monthly_Earnings_Savings_USD"]
    )

    # Normalize health
    data["Health_Normalized"] = (
        data["Health_Quality_Score_1to5"] / 5.0
    )

    # Normalize stress
    data["Stress_Normalized"] = (
        data["Stress_Level_1to5"] / 5.0
    )

    return data

In [43]:
X_processed = preprocess_data(X)

print("Processed shape:", X_processed.shape)

display(X_processed.head())

Processed shape: (55000, 13)


,Country,University,Country_HDI,Country_Life_Expectancy,Education_Level,Field_of_Study,Language_Proficiency,Health_Quality_Score_1to5,Stress_Level_1to5,Monthly_Earnings_Savings_USD,Earnings_Log,Health_Normalized,Stress_Normalized
0,Afghanistan,Afghan University,0.498,57.950001,Master's,Natural Sciences,Native/Fluent,2,2,1316.37,7.183393,0.4,0.4
1,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Education,Native/Fluent,2,4,1892.79,7.546335,0.4,0.8
2,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Social Sciences & Psychology,Native/Fluent,2,4,2030.31,7.616436,0.4,0.8
3,Afghanistan,Afghan University,0.498,57.950001,Undergraduate,Engineering & Technology,Native/Fluent,3,2,1646.31,7.406899,0.6,0.4
4,Afghanistan,Afghan University,0.498,57.950001,Master's,"Arts, Humanities & Design",Beginner (A1-A2),2,3,1299.43,7.170450,0.4,0.6


In [44]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 44000
Testing samples : 11000


In [45]:
# ============================================================
# CATBOOST REGRESSION
# ============================================================

cat_indices = [
    X_train.columns.get_loc(col)
    for col in CATEGORICAL_FEATURES
]

model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.04,

    loss_function="RMSE",
    eval_metric="RMSE",

    l2_leaf_reg=5,
    random_strength=1,

    random_seed=42,

    cat_features=cat_indices,

    od_type="Iter",
    od_wait=100,

    verbose=100,
    thread_count=-1
)

model.fit(
    X_train,
    y_train,

    eval_set=(X_test, y_test),

    use_best_model=True
)

0:	learn: 9.7783739	test: 9.7406390	best: 9.7406390 (0)	total: 46.6ms	remaining: 46.6s
100:	learn: 1.2898919	test: 1.2568676	best: 1.2568676 (100)	total: 3.12s	remaining: 27.8s
200:	learn: 1.1710634	test: 1.1488005	best: 1.1488005 (200)	total: 6.1s	remaining: 24.3s
300:	learn: 1.1548713	test: 1.1445779	best: 1.1445779 (300)	total: 8.93s	remaining: 20.7s
400:	learn: 1.1446949	test: 1.1430697	best: 1.1430697 (400)	total: 11.8s	remaining: 17.6s
500:	learn: 1.1363310	test: 1.1420231	best: 1.1420231 (500)	total: 14.8s	remaining: 14.7s
600:	learn: 1.1282428	test: 1.1414981	best: 1.1414651 (585)	total: 17.7s	remaining: 11.8s
700:	learn: 1.1211946	test: 1.1411511	best: 1.1411381 (696)	total: 20.7s	remaining: 8.83s
800:	learn: 1.1139871	test: 1.1411653	best: 1.1410477 (723)	total: 23.7s	remaining: 5.89s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.141047653
bestIteration = 723

Shrink model to first 724 iterations.


CatBoostRegressor(cat_features=[0, 1, 4, 5, 6], depth=8, eval_metric='RMSE', iterations=1000, l2_leaf_reg=5, learning_rate=0.04, loss_function='RMSE', od_type='Iter', od_wait=100, random_seed=42, random_strength=1, verbose=100)

In [46]:
# ============================================================
# MODEL EVALUATION
# ============================================================

predictions = model.predict(X_test)

# Keep predictions inside 0-100
predictions = np.clip(
    predictions,
    0,
    100
)

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

MODEL PERFORMANCE
MAE  : 0.9834
RMSE : 1.1410
R²   : 0.9872


In [47]:
# ============================================================
# PREDICTION ANALYSIS
# ============================================================

prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

prediction_df["Absolute_Error"] = (
    prediction_df["Actual"]
    -
    prediction_df["Predicted"]
).abs()

display(
    prediction_df.head(20)
)

,Actual,Predicted,Absolute_Error
0,54,53.939862,0.060138
1,59,57.224971,1.775029
2,65,66.437785,1.437785
3,62,63.349387,1.349387
4,64,65.057053,1.057053
5,47,46.424495,0.575505
6,69,70.380091,1.380091
7,56,55.468283,0.531717
8,64,65.309546,1.309546
9,76,76.878741,0.878741


In [48]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance = model.get_feature_importance()

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importance
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance
6,Language_Proficiency,23.565674
7,Health_Quality_Score_1to5,15.129640
8,Stress_Level_1to5,13.730183
11,Health_Normalized,12.443424
4,Education_Level,10.584238
10,Earnings_Log,9.493011
12,Stress_Normalized,8.754668
9,Monthly_Earnings_Savings_USD,5.274158
5,Field_of_Study,0.928242
0,Country,0.047802


In [49]:
# ============================================================
# BACKEND PREDICTION
# ============================================================

def predict_student(student_data):

    input_df = pd.DataFrame(
        [student_data]
    )

    processed = preprocess_data(
        input_df
    )

    prediction = model.predict(
        processed
    )[0]

    prediction = float(
        np.clip(
            prediction,
            0,
            100
        )
    )

    return round(
        prediction,
        2
    )

In [50]:
# ============================================================
# READINESS CATEGORY
# ============================================================

def get_readiness_status(score):

    if score >= 80:
        return "Highly Ready"

    elif score >= 65:
        return "Ready"

    elif score >= 50:
        return "Moderately Ready"

    else:
        return "Needs Improvement"

In [51]:
# ============================================================
# FINAL API RESPONSE
# ============================================================

def predict_student_api(student_data):

    score = predict_student(
        student_data
    )

    status = get_readiness_status(
        score
    )

    return {
        "success": True,

        "prediction": {
            "readiness_score": score,
            "readiness_status": status
        },

        "model": {
            "name": "Student Readiness AI",
            "version": "1.0.0"
        }
    }

In [52]:
# ============================================================
# TEST BACKEND PREDICTION
# ============================================================

sample_student = {
    "Country": "Afghanistan",
    "University": "Afghan University",
    "Country_HDI": 0.498,
    "Country_Life_Expectancy": 57.95,
    "Education_Level": "Master's",
    "Field_of_Study": "Natural Sciences",
    "Language_Proficiency": "Native/Fluent",
    "Health_Quality_Score_1to5": 2,
    "Stress_Level_1to5": 2,
    "Monthly_Earnings_Savings_USD": 1316.37
}

result = predict_student_api(
    sample_student
)

print(
    json.dumps(
        result,
        indent=4
    )
)

{
    "success": true,
    "prediction": {
        "readiness_score": 62.03,
        "readiness_status": "Moderately Ready"
    },
    "model": {
        "name": "Student Readiness AI",
        "version": "1.0.0"
    }
}


In [53]:
# ============================================================
# CREATE BACKEND ARTIFACT DIRECTORY
# ============================================================

OUTPUT_DIR = "/kaggle/working/backend_model"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print(OUTPUT_DIR)

/kaggle/working/backend_model


In [54]:
# ============================================================
# SAVE MODEL
# ============================================================

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "readiness_model.cbm"
)

model.save_model(
    MODEL_PATH
)

print(
    "Saved:",
    MODEL_PATH
)

Saved: /kaggle/working/backend_model/readiness_model.cbm


In [55]:
# ============================================================
# SAVE METADATA
# ============================================================

metadata = {

    "model_name":
        "Student Readiness AI",

    "version":
        "1.0.0",

    "model_type":
        "CatBoostRegressor",

    "target":
        TARGET,

    "target_range": [
        0,
        100
    ],

    "features":
        FEATURES,

    "categorical_features":
        CATEGORICAL_FEATURES,

    "numerical_features":
        NUMERICAL_FEATURES,

    "engineered_features": [
        "Earnings_Log",
        "Health_Normalized",
        "Stress_Normalized"
    ],

    "metrics": {
        "MAE": round(float(mae), 6),
        "RMSE": round(float(rmse), 6),
        "R2": round(float(r2), 6)
    }
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "metadata.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print("Metadata exported.")

Metadata exported.


In [56]:
# ============================================================
# EXPORT TEST PREDICTIONS
# ============================================================

test_predictions = X_test.copy()

test_predictions[
    "Actual_Readiness"
] = y_test.values

test_predictions[
    "Predicted_Readiness"
] = predictions

test_predictions[
    "Absolute_Error"
] = np.abs(
    test_predictions["Actual_Readiness"]
    -
    test_predictions["Predicted_Readiness"]
)

test_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "test_predictions.csv"
    ),
    index=False
)

print("Test predictions exported.")

Test predictions exported.


In [57]:
# ============================================================
# EXPORT FEATURE IMPORTANCE
# ============================================================

feature_importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "feature_importance.csv"
    ),
    index=False
)

print("Feature importance exported.")

Feature importance exported.


In [58]:
# ============================================================
# EXPORT SAMPLE RESPONSE
# ============================================================

sample_response = predict_student_api(
    sample_student
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "sample_prediction.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        sample_response,
        f,
        indent=4
    )

print("Sample response exported.")

Sample response exported.


In [59]:
# ============================================================
# CREATE BACKEND PREDICTOR
# ============================================================

predictor_code = r'''
import os
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor


BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)


MODEL_PATH = os.path.join(
    BASE_DIR,
    "readiness_model.cbm"
)


METADATA_PATH = os.path.join(
    BASE_DIR,
    "metadata.json"
)


# ------------------------------------------------------------
# LOAD METADATA
# ------------------------------------------------------------

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


FEATURES = metadata["features"]


# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

model = CatBoostRegressor()

model.load_model(
    MODEL_PATH
)


# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

def preprocess_data(data):

    data = data[FEATURES].copy()

    data["Earnings_Log"] = np.log1p(
        data["Monthly_Earnings_Savings_USD"]
    )

    data["Health_Normalized"] = (
        data["Health_Quality_Score_1to5"]
        / 5.0
    )

    data["Stress_Normalized"] = (
        data["Stress_Level_1to5"]
        / 5.0
    )

    return data


# ------------------------------------------------------------
# READINESS STATUS
# ------------------------------------------------------------

def get_readiness_status(score):

    if score >= 80:
        return "Highly Ready"

    elif score >= 65:
        return "Ready"

    elif score >= 50:
        return "Moderately Ready"

    return "Needs Improvement"


# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

def predict_student(student_data):

    input_df = pd.DataFrame(
        [student_data]
    )

    processed = preprocess_data(
        input_df
    )

    prediction = model.predict(
        processed
    )[0]

    prediction = float(
        np.clip(
            prediction,
            0,
            100
        )
    )

    return {
        "success": True,

        "prediction": {
            "readiness_score": round(
                prediction,
                2
            ),

            "readiness_status":
                get_readiness_status(
                    prediction
                )
        },

        "model": {
            "name":
                "Student Readiness AI",

            "version":
                "1.0.0"
        }
    }
'''


with open(
    os.path.join(
        OUTPUT_DIR,
        "predictor.py"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(
        predictor_code
    )

print("predictor.py created.")

predictor.py created.
